In [1]:
import cv2
import os
import numpy as np
import face_recognition as fr

In [ ]:

folder_path = "imgs"

for img_name in os.listdir(folder_path):
    img_path = os.path.join(folder_path, img_name)

    img_rgb = fr.load_image_file(img_path)
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)

    cv2.imshow("bgr", img_bgr)
    cv2.imshow("rgb", img_rgb)

    if cv2.waitKey(0) == ord('q'):
        break

cv2.waitKey(0)

In [ ]:
for img_name in os.listdir(folder_path):
    img_path = os.path.join(folder_path, img_name)

    imgelon = face_recognition.load_image_file(img_path)
    imgelon_rgb = cv2.cvtColor(imgelon, cv2.COLOR_BGR2RGB)

    faces = face_recognition.face_locations(imgelon_rgb)

    copy = imgelon_rgb.copy()

    for face in faces:
        cv2.rectangle(
            copy,
            (face[3], face[0]),
            (face[1], face[2]),
            (255, 0, 255),
            2
        )

    cv2.imshow("original", imgelon_rgb)
    cv2.imshow("copy", copy)

    if cv2.waitKey(0) == ord('q'):
        break

In [ ]:
train_encodings = []

for img_name in os.listdir(folder_path):
    img_path = os.path.join(folder_path, img_name)

    imgelon = face_recognition.load_image_file(img_path)
    imgelon_rgb = cv2.cvtColor(imgelon, cv2.COLOR_BGR2RGB)

    encodings = face_recognition.face_encodings(imgelon_rgb)

    if len(encodings) > 0:
        train_encodings.append(encodings[0])

print(len(train_encodings))

In [ ]:

test_folder = "imgs"

for img_name in os.listdir(test_folder):
    img_path = os.path.join(test_folder, img_name)

    test = face_recognition.load_image_file(img_path)
    test = cv2.cvtColor(test, cv2.COLOR_BGR2RGB)

    encodings = face_recognition.face_encodings(test)
    if len(encodings) == 0:
        continue

    test_encode = encodings[0]
    results = face_recognition.compare_faces(train_encodings, test_encode)

    print(img_name, results)

In [ ]:
import cv2
import face_recognition as fr
import os
import numpy as np
from datetime import datetime
import pickle

In [ ]:
path = 'imgs'
images = []
classNames = []mylist = os.listdir(path)
for cl in mylist:
    curImg = cv2.imread(f'{path}/{cl}')
    images.append(curImg)
    classNames.append(os.path.splitext(cl)[0])

In [ ]:
def findEncodings(images):
    encodeList = []
    for img in images:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        encoded_face = face_recognition.face_encodings(img)[0]
        encodeList.append(encoded_face)
    return encodeList
encoded_face_train = findEncodings(images)

In [ ]:
def markAttendance(name):
    with open('Attendance.csv','r+') as f:
        myDataList = f.readlines()
        nameList = []
        for line in myDataList:
            entry = line.split(',')
            nameList.append(entry[0])
        if name not in nameList:
            now = datetime.now()
            time = now.strftime('%I:%M:%S:%p')
            date = now.strftime('%d-%B-%Y')
            f.writelines(f'n{name}, {time}, {date}')

In [ ]:
# take pictures from webcam 
cap  = cv2.VideoCapture(0)while True:
    success, img = cap.read()
    imgS = cv2.resize(img, (0,0), None, 0.25,0.25)
    imgS = cv2.cvtColor(imgS, cv2.COLOR_BGR2RGB)
    faces_in_frame = face_recognition.face_locations(imgS)
    encoded_faces = face_recognition.face_encodings(imgS, faces_in_frame)for encode_face, faceloc in zip(encoded_faces,faces_in_frame):
        matches = face_recognition.compare_faces(encoded_face_train, encode_face)
        faceDist = face_recognition.face_distance(encoded_face_train, encode_face)
        matchIndex = np.argmin(faceDist)
        print(matchIndex)
        if matches[matchIndex]:
            name = classNames[matchIndex].upper().lower()
            y1,x2,y2,x1 = faceloc
            # since we scaled down by 4 times
            y1, x2,y2,x1 = y1*4,x2*4,y2*4,x1*4
            cv2.rectangle(img,(x1,y1),(x2,y2),(0,255,0),2)
            cv2.rectangle(img, (x1,y2-35),(x2,y2), (0,255,0), cv2.FILLED)
            cv2.putText(img,name, (x1+6,y2-5), cv2.FONT_HERSHEY_COMPLEX,1,(255,255,255),2)
            markAttendance(name)
    cv2.imshow('webcam', img)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break